In [1]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

# Import rpy2 để sử dụng R
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

print("Đang kiểm tra và cài đặt thư viện R...")

# Import các thư viện R cần thiết
base = importr('base')
utils = importr('utils')

# Kiểm tra và cài đặt rugarch
try:
    rugarch = importr('rugarch')
    print("✓ rugarch đã được load")
except:
    print("⚠ Đang cài đặt rugarch (có thể mất vài phút)...")
    utils.install_packages('rugarch', repos='https://cloud.r-project.org')
    rugarch = importr('rugarch')
    print("✓ rugarch đã được cài đặt và load")

# Load dữ liệu
df = pd.read_csv("D:\\g2data_asymmetric_final.csv")
df.rename(columns={'date': 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

print(f"\n✓ Dữ liệu đã load: {df.shape}")
print(f"✓ Khoảng thời gian: {df.index[0]} đến {df.index[-1]}")
print(f"\n✓ rpy2 và rugarch sẵn sàng để sử dụng!")

Đang kiểm tra và cài đặt thư viện R...
✓ rugarch đã được load

✓ Dữ liệu đã load: (3845, 25)
✓ Khoảng thời gian: 2010-10-01 00:00:00 đến 2025-06-30 00:00:00

✓ rpy2 và rugarch sẵn sàng để sử dụng!


In [2]:
df.columns

Index(['dlog_GoldPrice', 'dlog_VNIndex_pos', 'dlog_VNIndex_neg',
       'dlog_OilPrice_pos', 'dlog_OilPrice_neg', 'dlog_ExchangeRate_pos',
       'dlog_ExchangeRate_neg', 'd_CPI_pos', 'd_CPI_neg', 'd_IR_pos',
       'd_IR_neg', 'dlog_GoldReserve_pos', 'dlog_GoldReserve_neg',
       'dlog_M2_pos', 'dlog_M2_neg', 'dlog_GoldDemand_pos',
       'dlog_GoldDemand_neg', 'dlog_VNIndex', 'dlog_OilPrice',
       'dlog_ExchangeRate', 'd_CPI', 'd_IR', 'dlog_M2', 'dlog_GoldReserve',
       'dlog_GoldDemand'],
      dtype='object')

## Hàm ARX(1)-EGARCHX(1,1)

Hàm này sẽ:
1. Lọc dữ liệu theo start_period và end_period
2. Tạo các biến lag 1 cho AR part (GAP_lag1 và các biến gốc lag 1)
3. Sử dụng các biến _pos và _neg cho EGARCH part
4. Fit mô hình EGARCH(1,1) với mean model là ARX(1)

In [3]:
def run_arx_egarchx_model(start_period, end_period, selected_vars=None):
    """
    Chạy mô hình ARX(1)-EGARCHX(1,1) trên window dữ liệu được chỉ định
    Sử dụng rugarch (R) thông qua rpy2 để hỗ trợ external regressors trong variance equation
    
    Parameters:
    -----------
    start_period : str
        Ngày bắt đầu (format: 'YYYY-MM-DD')
    end_period : str
        Ngày kết thúc (format: 'YYYY-MM-DD')
    selected_vars : list, optional
        Danh sách biến để sử dụng (default: None = dùng tất cả)
        VD: ['dlog_OilPrice', 'dlog_ExchangeRate', 'd_CPI']
    
    Returns:
    --------
    dict : Dictionary chứa kết quả mô hình và thông tin
    
    Note:
    -----
    Mean Equation (ARX): GAP_t = const + β1*GAP_{t-1} + Σ(β_i * X_{i,t-1})
    Variance Equation (EGARCHX): log(σ²_t) = ω + α*|z_{t-1}| + γ*z_{t-1} + β*log(σ²_{t-1}) + Σ(δ_i * Exog_i)
    """
    
    print("="*80)
    print(f"Chạy mô hình ARX(1)-EGARCHX(1,1) với rugarch (R)")
    print(f"Window: {start_period} đến {end_period}")
    print("="*80)
    
    # 1. Lọc dữ liệu theo window
    df_window = df.loc[start_period:end_period].copy()
    print(f"\n✓ Số quan sát trong window: {len(df_window)}")
    
    # 2. Định nghĩa các biến
    original_vars = ['dlog_OilPrice', 'dlog_ExchangeRate', 'dlog_VNIndex', 'dlog_GoldDemand', 
                     'd_CPI', 'dlog_GoldReserve', 'dlog_M2', 'd_IR']
    
    # 3. Tạo lag 1 cho AR part
    df_window['dlog_GoldPrice_lag1'] = df_window['dlog_GoldPrice'].shift(1)
    for var in original_vars:
        df_window[f'{var}_lag1'] = df_window[var].shift(1)
    
    # 4. Loại bỏ missing values
    df_model = df_window.dropna().reset_index(drop=True)
    print(f"✓ Số quan sát sau khi loại bỏ missing: {len(df_model)}")
    
    # 5. Chuẩn bị biến
    y = df_model['dlog_GoldPrice'].values
    
    # Mean equation: dlog_GoldPrice_lag1 + các biến gốc lag 1
    mean_vars = ['dlog_GoldPrice_lag1'] + [f'{var}_lag1' for var in original_vars]
    X_mean = df_model[mean_vars]
    
    # Variance equation: các biến _10 và _90 hoặc _pos và _neg
    var_exog_vars = []
    for var in original_vars:
        # var_exog_vars.extend([f'{var}_10', f'{var}_90'])
        var_exog_vars.extend([f'{var}_pos', f'{var}_neg'])
    X_var = df_model[var_exog_vars]
    
    # KIỂM TRA VÀ LOẠI BỎ CÁC BIẾN CÓ VARIANCE = 0 hoặc quá thấp
    print(f"\n🔍 Kiểm tra variance của external regressors...")
    var_variances = X_var.var()
    zero_var_cols = var_variances[var_variances == 0].index.tolist()
    low_var_cols = var_variances[(var_variances > 0) & (var_variances < 1e-10)].index.tolist()
    
    if zero_var_cols:
        print(f"⚠️ Loại bỏ {len(zero_var_cols)} biến có variance = 0:")
        for col in zero_var_cols:
            print(f"   - {col}")
        X_var = X_var.drop(columns=zero_var_cols)
        var_exog_vars = [v for v in var_exog_vars if v not in zero_var_cols]
    
    if low_var_cols:
        print(f"⚠️ Loại bỏ {len(low_var_cols)} biến có variance < 1e-10:")
        for col in low_var_cols:
            print(f"   - {col} (var = {var_variances[col]:.2e})")
        X_var = X_var.drop(columns=low_var_cols)
        var_exog_vars = [v for v in var_exog_vars if v not in low_var_cols]
    
    print(f"\n✓ Mean equation: {X_mean.shape[1]} biến (dlog_GoldPrice_lag1 + {len(original_vars)} biến lag 1)")
    print(f"✓ Variance equation: {X_var.shape[1]} external regressors sau khi loại bỏ biến problematic")
    
    # 6. Chuyển dữ liệu sang R
    print("\n" + "="*80)
    print("Đang chuẩn bị dữ liệu cho R...")
    print("="*80)
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['y_r'] = ro.FloatVector(y)
        ro.globalenv['X_mean_r'] = pandas2ri.py2rpy(X_mean)
        ro.globalenv['X_var_r'] = pandas2ri.py2rpy(X_var)
    
    # 7. Specify model trong R
    print("\n✓ Đang specify mô hình EGARCH(1,1) với external regressors...")
    
    try:
        ro.r('''
        # Specify EGARCH model với external regressors trong variance
        spec <- ugarchspec(
            mean.model = list(armaOrder = c(0, 0), 
                             include.mean = TRUE,
                             external.regressors = as.matrix(X_mean_r)),
            variance.model = list(model = "eGARCH", 
                                garchOrder = c(1, 1),
                                external.regressors = as.matrix(X_var_r)),
            distribution.model = "norm"
        )
        ''')
        
        print("✓ Model specification thành công!")
        
        # 8. Fit model
        print("✓ Đang fit mô hình (có thể mất vài phút)...")
        
        ro.r('''
        # Fit model với solver hybrid để tăng độ ổn định
        fit <- tryCatch({
            ugarchfit(spec = spec, data = y_r, solver = "hybrid")
        }, error = function(e) {
            # Nếu hybrid fail, thử solver khác
            ugarchfit(spec = spec, data = y_r, solver = "solnp")
        })
        ''')
        
        print("\n✓ Mô hình đã được fit thành công!")
        
    except Exception as e:
        print(f"\n✗ Lỗi khi fit model: {e}")
        print("Đang thử với configuration đơn giản hơn...")
        return None
    
    # 9. Extract kết quả từ R
    print("\n" + "="*80)
    print("Đang extract kết quả từ R...")
    print("="*80)
    
    # Kiểm tra xem model có converged không
    try:
        convergence = int(ro.r('fit@fit$convergence')[0])
        if convergence != 0:
            print(f"\n⚠ WARNING: Model không converge hoàn toàn (convergence code: {convergence})")
            print("⚠ Kết quả có thể không đáng tin cậy. Vẫn tiếp tục extract kết quả...")
    except:
        pass
    
    # Lấy thông tin từ model - sử dụng cách an toàn hơn
    try:
        ro.r('''
        # Extract coefficients và standard errors
        coef_vals <- coef(fit)
        
        # Lấy matcoef (không robust)
        mat_coef <- fit@fit$matcoef
        
        # Kiểm tra nếu matcoef NULL hoặc có vấn đề
        if (is.null(mat_coef) || nrow(mat_coef) == 0) {
            results_table <- NULL
        } else {
            # Tạo results table - chú ý: tên cột có space ở đầu
            results_table <- data.frame(
                Parameter = names(coef_vals),
                Coefficient = as.numeric(coef_vals),
                Std_Error = mat_coef[, " Std. Error"],
                T_Value = mat_coef[, " t value"],
                P_Value = mat_coef[, "Pr(>|t|)"]
            )
            rownames(results_table) <- NULL
        }
        ''')
        
        # Convert kết quả sang pandas - lấy trực tiếp từ R
        results_table = ro.r('results_table')
        
        # Kiểm tra nếu results_table là NULL
        if str(type(results_table)) == "<class 'rpy2.rinterface_lib.sexp.NULLType'>":
            print("\n✗ CRITICAL: Model fit thất bại, không có kết quả.")
            print("✗ Nguyên nhân có thể do: dữ liệu không đủ, quá nhiều biến, hoặc vấn đề numerical stability")
            return None
        
        # Convert sang pandas DataFrame
        with localconverter(ro.default_converter + pandas2ri.converter):
            results_table = pd.DataFrame({
                'Parameter': list(results_table.rx2('Parameter')),
                'Coefficient': list(results_table.rx2('Coefficient')),
                'Std_Error': list(results_table.rx2('Std_Error')),
                'T_Value': list(results_table.rx2('T_Value')),
                'P_Value': list(results_table.rx2('P_Value'))
            })
    except Exception as e:
        print(f"\n✗ Lỗi khi extract kết quả: {e}")
        return None
    
    # Lấy diagnostic tests
    try:
        ro.r('''
        # Lấy residuals
        std_res <- as.numeric(residuals(fit, standardize=TRUE))
        
        # 1. Ljung-Box test cho Autocorrelation (Lag 10)
        lb_test <- Box.test(std_res, lag=10, type="Ljung-Box")
        
        # 2. ARCH-LM test cho gián đoạn phương sai (Heteroskedasticity) qua Ljung-Box bình phương (Lag 10)
        lb_sq_test <- Box.test(std_res^2, lag=10, type="Ljung-Box")
        
        # 3. Engle-Ng Sign Bias Test
        sb_obj <- signbias(fit)
        sb_p <- as.numeric(sb_obj[4, "prob"]) # Lấy Joint Effect p-value ở dòng 4, cột prob
        
        diag_df <- data.frame(
            Test = c("Ljung-Box Q(10) (Autocorrelation)", "ARCH-LM Q^2(10) (Heteroskedasticity)", "Engle-Ng Sign Bias (Joint Effect)"),
            P_Value = c(as.numeric(lb_test$p.value), as.numeric(lb_sq_test$p.value), sb_p)
        )
        ''')
        diag_df = ro.r('diag_df')
        with localconverter(ro.default_converter + pandas2ri.converter):
            diag_df = pd.DataFrame({
                'Test': list(diag_df.rx2('Test')),
                'P_Value': list(diag_df.rx2('P_Value'))
            })
    except Exception as e:
        print(f"\n⚠ Không thể extract diagnostic tests: {e}")
        diag_df = None

    # Lấy thông tin mô hình
    try:
        aic = float(ro.r('infocriteria(fit)["Akaike",]')[0])
        bic = float(ro.r('infocriteria(fit)["Bayes",]')[0])
        loglik = float(ro.r('likelihood(fit)')[0])
    except:
        aic = bic = loglik = np.nan
    
    # 10. Hiển thị kết quả
    print("\n" + "="*100)
    print("T-VALUES VÀ P-VALUES CHI TIẾT CHO TỪNG BIẾN")
    print("="*100)
    print("Phương pháp: Quasi-Maximum Likelihood Estimation")
    print("="*100)
    
    # Tách Mean và Variance parameters
    mean_mask = results_table['Parameter'].str.contains('mu|mxreg', na=False)
    var_mask = ~mean_mask
    
    mean_params = results_table[mean_mask].reset_index(drop=True)
    var_params = results_table[var_mask].reset_index(drop=True)
    
    # 11. Hiển thị Mean Equation
    print("\n" + "="*100)
    print("MEAN EQUATION (ARX PART): dlog_GoldPrice_t = μ + Σ(β_i * X_{i,t-1})")
    print("="*100)
    print(f"{'Parameter':<30} | {'Coefficient':>15} | {'Std Error':>15} | {'T-Value':>12} | {'P-Value':>12} | {'Sig'}")
    print("-" * 100)
    
    # Map tên parameters cho mean equation
    for idx, row in mean_params.iterrows():
        param_original = row['Parameter']
        if param_original == 'mu':
            param_name = 'Const'
        elif 'mxreg' in param_original:
            mxreg_idx = int(param_original.replace('mxreg', '')) - 1
            param_name = mean_vars[mxreg_idx] if mxreg_idx < len(mean_vars) else param_original
        else:
            param_name = param_original
            
        sig_mark = "***" if row['P_Value'] < 0.01 else "**" if row['P_Value'] < 0.05 else "*" if row['P_Value'] < 0.1 else ""
        print(f"{param_name:<30} | {row['Coefficient']:15.8f} | {row['Std_Error']:15.8f} | {row['T_Value']:12.6f} | {row['P_Value']:12.8f} | {sig_mark:>3}")
    
    # 12. Hiển thị Variance Equation
    print("\n" + "="*100)
    print("VARIANCE EQUATION (EGARCHX PART): log(σ²_t) = ω + α*|z_{t-1}| + γ*z_{t-1} + β*log(σ²_{t-1}) + Σ(δ_i * Exog_i)")
    print("="*100)
    print(f"{'Parameter':<30} | {'Coefficient':>15} | {'Std Error':>15} | {'T-Value':>12} | {'P-Value':>12} | {'Sig'}")
    print("-" * 100)
    
    # Hiển thị variance parameters
    for idx, row in var_params.iterrows():
        param_original = row['Parameter']
        param_name = param_original
        
        # Map tên cho external regressors
        if 'vxreg' in param_original:
            vxreg_idx = int(param_original.replace('vxreg', '')) - 1
            if vxreg_idx < len(var_exog_vars):
                param_name = var_exog_vars[vxreg_idx]
        
        sig_mark = "***" if row['P_Value'] < 0.01 else "**" if row['P_Value'] < 0.05 else "*" if row['P_Value'] < 0.1 else ""
        print(f"{param_name:<30} | {row['Coefficient']:15.8f} | {row['Std_Error']:15.8f} | {row['T_Value']:12.6f} | {row['P_Value']:12.8f} | {sig_mark:>3}")
    
    # 13. Ghi chú
    print("\n" + "="*100)
    print("GHI CHÚ:")
    print("  - T-value = Coefficient / Standard_Error")
    print("  - P-value: Two-tailed test với H₀: β = 0")
    print("  - Mức ý nghĩa: *** p<0.01, ** p<0.05, * p<0.1")
    print("  - omega: Constant term trong variance equation")
    print("  - alpha1: Magnitude effect (ARCH term)")
    print("  - gamma1: Asymmetric/leverage effect")
    print("  - beta1: Persistence effect (GARCH term)")
    print("  - _pos/_neg: External regressors trong variance (asymmetric effects)")
    print("="*100)
    
    # 14. Thống kê tổng quan
    sig_01_mean = (mean_params['P_Value'] < 0.01).sum()
    sig_05_mean = (mean_params['P_Value'] < 0.05).sum()
    sig_10_mean = (mean_params['P_Value'] < 0.10).sum()
    
    sig_01_var = (var_params['P_Value'] < 0.01).sum()
    sig_05_var = (var_params['P_Value'] < 0.05).sum()
    sig_10_var = (var_params['P_Value'] < 0.10).sum()
    
    print(f"\nTỔNG QUAN Ý NGHĨA THỐNG KÊ:")
    print(f"\nMean Equation (ARX):")
    print(f"  - p < 0.01: {sig_01_mean}/{len(mean_params)} biến")
    print(f"  - p < 0.05: {sig_05_mean}/{len(mean_params)} biến")
    print(f"  - p < 0.10: {sig_10_mean}/{len(mean_params)} biến")
    
    print(f"\nVariance Equation (EGARCHX):")
    print(f"  - p < 0.01: {sig_01_var}/{len(var_params)} parameters")
    print(f"  - p < 0.05: {sig_05_var}/{len(var_params)} parameters")
    print(f"  - p < 0.10: {sig_10_var}/{len(var_params)} parameters")
    
    # 15. In Diagnostic Tests
    if diag_df is not None:
        print("\n" + "="*100)
        print("ROBUSTNESS & DIAGNOSTIC TESTS (P-values > 0.05 là mô hình tốn/vững)")
        print("="*100)
        for idx, row in diag_df.iterrows():
            status = "PASS (Robust)" if row['P_Value'] > 0.05 else "FAIL (Cần xem xét)"
            print(f"  - {row['Test']:<40}: p-value = {row['P_Value']:.4f} => {status}")
    
    # 16. Trả về kết quả
    results_dict = {
        'window': (start_period, end_period),
        'n_obs': len(df_model),
        'n_vars': len(original_vars),
        'selected_vars': original_vars,
        'results_table': results_table,
        'mean_params': mean_params,
        'var_params': var_params,
        'diag_tests': diag_df,
        'aic': aic,
        'bic': bic,
        'loglikelihood': loglik,
        'fit_object': ro.r('fit')  # R object
    }
    
    print("\n" + "="*100)
    print("THÔNG TIN MÔ HÌNH")
    print("="*100)
    print(f"Số quan sát: {len(df_model)}")
    print(f"Số biến: {len(original_vars)}")
    print(f"AIC: {aic:.4f}")
    print(f"BIC: {bic:.4f}")
    print(f"Log-Likelihood: {loglik:.4f}")
    print("="*100 + "\n")
    
    return results_dict

## Hàm cho 3 Windows cụ thể

Tạo 3 hàm wrapper cho các window:
- Window 1: 2011-2014
- Window 2: 2014-2020
- Window 3: 2020-hết

In [4]:
def run_model_window1():
    """
    Chạy mô hình cho Window 1: 2010-2014
    """
    return run_arx_egarchx_model('2010-06-01', '2014-12-31')


def run_model_window2():
    """
    Chạy mô hình cho Window 2: 2015-2020
    """
    return run_arx_egarchx_model('2015-01-01', '2019-12-31')


def run_model_window3():
    """
    Chạy mô hình cho Window 3: 2020-hết
    """
    # Lấy ngày cuối cùng trong dataset
    last_date = df.index[-1].strftime('%Y-%m-%d')
    return run_arx_egarchx_model('2020-01-01', last_date)

In [6]:
# KIỂM TRA DỮ LIỆU TRƯỚC KHI CHẠY MODEL
print("="*80)
print("KIỂM TRA DỮ LIỆU")
print("="*80)

# Kiểm tra dữ liệu tổng quan
print(f"\n📊 Thông tin dataset:")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0]} đến {df.index[-1]}")
print(f"\n📋 Columns: {df.columns.tolist()}")

# Kiểm tra missing values
print(f"\n❗ Missing values:")
print(df.isnull().sum())

# Kiểm tra các biến gốc
print(f"\n📈 Thống kê các biến gốc:")
original_vars = ['dlog_GoldPrice', 'dlog_OilPrice', 'dlog_ExchangeRate', 'dlog_VNIndex',  
                 'd_CPI', 'd_IR', 'dlog_M2']
print(df[original_vars].describe())

# Kiểm tra dữ liệu trong window 
print(f"\n🔍 Kiểm tra thử Window  (2011-01-01 đến 2016-12-31):")
df_w1 = df.loc['2011-01-01':'2016-12-31']
print(f"Số quan sát: {len(df_w1)}")
print(f"Missing values trong window 1:")
print(df_w1[original_vars].isnull().sum())
print(f"\nSố dòng có ít nhất 1 NaN: {df_w1[original_vars].isnull().any(axis=1).sum()}")
print(f"Số dòng đầy đủ: {df_w1[original_vars].notna().all(axis=1).sum()}")

# Kiểm tra các biến _pos và _neg
print(f"\n🔄 Kiểm tra biến _pos và _neg:")
pos_neg_vars = [col for col in df.columns if '_pos' in col or '_neg' in col]
print(f"Số biến _pos/_neg: {len(pos_neg_vars)}")
print(f"\nCác biến luôn = 0 (problematic):")
for var in pos_neg_vars:
    if (df[var] == 0).all():
        print(f"  ⚠ {var}: ALL ZEROS")
    elif (df[var] == 0).sum() / len(df) > 0.99:
        print(f"  ⚠ {var}: {(df[var] == 0).sum() / len(df) * 100:.1f}% zeros")

KIỂM TRA DỮ LIỆU

📊 Thông tin dataset:
Shape: (3845, 25)
Date range: 2010-10-01 00:00:00 đến 2025-06-30 00:00:00

📋 Columns: ['dlog_GoldPrice', 'dlog_VNIndex_pos', 'dlog_VNIndex_neg', 'dlog_OilPrice_pos', 'dlog_OilPrice_neg', 'dlog_ExchangeRate_pos', 'dlog_ExchangeRate_neg', 'd_CPI_pos', 'd_CPI_neg', 'd_IR_pos', 'd_IR_neg', 'dlog_GoldReserve_pos', 'dlog_GoldReserve_neg', 'dlog_M2_pos', 'dlog_M2_neg', 'dlog_GoldDemand_pos', 'dlog_GoldDemand_neg', 'dlog_VNIndex', 'dlog_OilPrice', 'dlog_ExchangeRate', 'd_CPI', 'd_IR', 'dlog_M2', 'dlog_GoldReserve', 'dlog_GoldDemand']

❗ Missing values:
dlog_GoldPrice           0
dlog_VNIndex_pos         0
dlog_VNIndex_neg         0
dlog_OilPrice_pos        0
dlog_OilPrice_neg        0
dlog_ExchangeRate_pos    0
dlog_ExchangeRate_neg    0
d_CPI_pos                0
d_CPI_neg                0
d_IR_pos                 0
d_IR_neg                 0
dlog_GoldReserve_pos     0
dlog_GoldReserve_neg     0
dlog_M2_pos              0
dlog_M2_neg              0
dlog_

In [8]:
# Chạy lại model với hàm đã fix
print("Chạy mô hình Window 1 với rugarch (đã fix lỗi)...")
# Chạy Window 1: 2010- Dec 2014
results_w1 = run_model_window1()

Chạy mô hình Window 1 với rugarch (đã fix lỗi)...
Chạy mô hình ARX(1)-EGARCHX(1,1) với rugarch (R)
Window: 2010-06-01 đến 2014-12-31

✓ Số quan sát trong window: 1109
📊 Sử dụng 8 biến: dlog_OilPrice, dlog_ExchangeRate, dlog_VNIndex, dlog_GoldDemand, d_CPI, dlog_GoldReserve, dlog_M2, d_IR
✓ Số quan sát sau khi loại bỏ missing: 1108

🔍 Kiểm tra variance của external regressors...

✓ Mean equation: 9 biến (dlog_GoldPrice_lag1 + 7 biến lag 1)
✓ Variance equation: 16 external regressors sau khi loại bỏ biến problematic

Đang chuẩn bị dữ liệu cho R...

✓ Đang specify mô hình EGARCH(1,1) với external regressors...
✓ Model specification thành công!
✓ Đang fit mô hình (có thể mất vài phút)...

✓ Mô hình đã được fit thành công!

Đang extract kết quả từ R...

T-VALUES VÀ P-VALUES CHI TIẾT CHO TỪNG BIẾN
Phương pháp: Quasi-Maximum Likelihood Estimation

MEAN EQUATION (ARX PART): dlog_GoldPrice_t = μ + Σ(β_i * X_{i,t-1})
Parameter                      |     Coefficient |       Std Error |      T-Val

In [9]:
# Chạy Window 2: 2015- Dec 2020
results_w2 = run_model_window2()

Chạy mô hình ARX(1)-EGARCHX(1,1) với rugarch (R)
Window: 2015-01-01 đến 2019-12-31

✓ Số quan sát trong window: 1825
✓ Số quan sát sau khi loại bỏ missing: 1824

🔍 Kiểm tra variance của external regressors...
⚠️ Loại bỏ 1 biến có variance = 0:
   - d_IR_pos

✓ Mean equation: 9 biến (dlog_GoldPrice_lag1 + 7 biến lag 1)
✓ Variance equation: 15 external regressors sau khi loại bỏ biến problematic

Đang chuẩn bị dữ liệu cho R...

✓ Đang specify mô hình EGARCH(1,1) với external regressors...
✓ Model specification thành công!
✓ Đang fit mô hình (có thể mất vài phút)...

✓ Mô hình đã được fit thành công!

Đang extract kết quả từ R...

T-VALUES VÀ P-VALUES CHI TIẾT CHO TỪNG BIẾN
Phương pháp: Quasi-Maximum Likelihood Estimation

MEAN EQUATION (ARX PART): dlog_GoldPrice_t = μ + Σ(β_i * X_{i,t-1})
Parameter                      |     Coefficient |       Std Error |      T-Value |      P-Value | Sig
---------------------------------------------------------------------------------------------------

In [10]:
# 2020 - Aug 2025
print("="*80)
print("CHẠY LẠI WINDOW 3 với logic loại bỏ biến variance = 0")
print("="*80)
results_w3_fixed = run_model_window3()

CHẠY LẠI WINDOW 3 với logic loại bỏ biến variance = 0
Chạy mô hình ARX(1)-EGARCHX(1,1) với rugarch (R)
Window: 2020-01-01 đến 2025-06-30

✓ Số quan sát trong window: 1432
📊 Sử dụng 8 biến: dlog_OilPrice, dlog_ExchangeRate, dlog_VNIndex, dlog_GoldDemand, d_CPI, dlog_GoldReserve, dlog_M2, d_IR
✓ Số quan sát sau khi loại bỏ missing: 1431

🔍 Kiểm tra variance của external regressors...

✓ Mean equation: 9 biến (dlog_GoldPrice_lag1 + 7 biến lag 1)
✓ Variance equation: 16 external regressors sau khi loại bỏ biến problematic

Đang chuẩn bị dữ liệu cho R...

✓ Đang specify mô hình EGARCH(1,1) với external regressors...
✓ Model specification thành công!
✓ Đang fit mô hình (có thể mất vài phút)...

✓ Mô hình đã được fit thành công!

Đang extract kết quả từ R...

T-VALUES VÀ P-VALUES CHI TIẾT CHO TỪNG BIẾN
Phương pháp: Quasi-Maximum Likelihood Estimation

MEAN EQUATION (ARX PART): dlog_GoldPrice_t = μ + Σ(β_i * X_{i,t-1})
Parameter                      |     Coefficient |       Std Error |      T